# RQ2 parameter-exposure Pareto preview — CPU only

This notebook computes exact loss-gradient exposure from the repository's nested-prefix modules, enumerates all 91 endpoint-locked four-anchor sets, and previews the new $(R_G,D_E)$ frontier. It does not load checkpoints, CIFAR-100, predictions, specialization gaps, accuracy, or test data, and it does not freeze a Hybrid-v3 choice.

## Secure checkout
Use Kaggle secret `github_token`. Select CPU/accelerator None. Internet is used only to clone the private repository.

In [ ]:
import os, subprocess, sys, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

## Locate the completed RQ2-v1 development artifact

In [ ]:
import importlib
import rq2_anchor_placement
import rq2_parameter_exposure
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
rq2_parameter_exposure = importlib.reload(rq2_parameter_exposure)
RQ2_INPUT_ROOT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT_ROOT.exists(), f'Attach notebook output: {RQ2_INPUT_ROOT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(
    RQ2_INPUT_ROOT, '/kaggle/working/materialized-rq2-exposure'
)
print('Validated RQ2-v1 root:', RQ2_ROOT)

## Enumerate exact exposure and preview the frontier
`D_E` is the average lost number of Uniform anchor-loss paths per trainable scalar parameter. The module also exports elastic-only, per-layer, activation-band, and raw-width/FLOPs proxy diagnostics.

In [ ]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
OUTPUT_DIR = Path('/kaggle/working/rq2-parameter-exposure-preview')
started = time.perf_counter()
result = rq2_parameter_exposure.run_parameter_exposure_preview(RQ2_ROOT, OUTPUT_DIR)
print(f'Completed in {time.perf_counter() - started:.2f} seconds')
print(json.dumps(result, indent=2))

## Inspect the mechanism and new Pareto frontier

In [ ]:
display(Markdown((OUTPUT_DIR / 'parameter_exposure_report.md').read_text()))
display(Markdown('### Uniform / PureGeo-v1 / prior Hybrid-v2'))
display(pd.read_csv(OUTPUT_DIR / 'parameter_exposure_reference_sets.csv'))
display(Markdown('### Exposure loss by actual layer'))
display(pd.read_csv(OUTPUT_DIR / 'parameter_exposure_reference_layers.csv'))
display(Markdown('### New (R_G, D_E) Pareto frontier'))
display(pd.read_csv(OUTPUT_DIR / 'parameter_exposure_pareto_RG_DE.csv'))
display(Markdown('### Is exposure only a raw-spacing/FLOPs proxy?'))
display(pd.read_csv(OUTPUT_DIR / 'parameter_exposure_proxy_correlations.csv'))
display(Image(filename=str(OUTPUT_DIR / 'parameter_exposure_pareto.png')))
display(Image(filename=str(OUTPUT_DIR / 'parameter_activation_bands.png')))

## Validate and export the CPU bundle

In [ ]:
REQUIRED = [
    'parameter_active_counts_by_width_layer.csv',
    'parameter_activation_bands.csv',
    'parameter_exposure_all_anchor_sets.csv',
    'parameter_exposure_by_anchor_set_layer.csv',
    'parameter_exposure_pareto_RG_DE.csv',
    'parameter_exposure_pareto_RG_RR_DE.csv',
    'parameter_exposure_reference_sets.csv',
    'parameter_exposure_reference_layers.csv',
    'parameter_exposure_proxy_correlations.csv',
    'parameter_exposure_pareto.png',
    'parameter_activation_bands.png',
    'parameter_exposure_report.md',
    'parameter_exposure_preview.json',
]
missing = [name for name in REQUIRED if not (OUTPUT_DIR / name).is_file()]
assert not missing, f'Missing outputs: {missing}'
candidates = pd.read_csv(OUTPUT_DIR / 'parameter_exposure_all_anchor_sets.csv')
assert len(candidates) == 91
assert candidates.loc[candidates.is_uniform, 'D_E'].iloc[0] == 0.0
bundle_path = Path('/kaggle/working/rq2-parameter-exposure-preview.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
print('Download:', bundle_path)
bundle_path